In [14]:
import pandas as pd
import requests
from sentence_transformers import SentenceTransformer, util
from thefuzz import fuzz

from kalshi_client import KalshiClient
from polymarket_client import PolymarketClient
from utils import create_match_keys, clean_text_cols

In [15]:
# Initialize Clients and Model
kalshi = KalshiClient()
poly = PolymarketClient()
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5448.59it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
## --- 1. Generic Matching Engine ---

import pandas as pd
from sentence_transformers import util
from thefuzz import fuzz

def get_matches(df_k, df_p, text_col, id_col, type_label='event', threshold=0.65):
    """
    Refined Blocking Matcher:
    Uses a simple average of Semantic Score and Fuzzy Title Similarity 
    to determine the Total Score.
    """
    common_keys = set(df_k['match_key']) & set(df_p['match_key'])
    all_results = []

    for key in common_keys:
        k_subset = df_k[df_k['match_key'] == key].reset_index(drop=True)
        p_subset = df_p[df_p['match_key'] == key].reset_index(drop=True)
        
        if k_subset.empty or p_subset.empty:
            continue

        # 1. Block-level Vectorization
        k_titles = k_subset[text_col].astype(str).tolist()
        p_titles = p_subset[text_col].astype(str).tolist()
        
        k_embeddings = model.encode(k_titles, convert_to_tensor=True)
        p_embeddings = model.encode(p_titles, convert_to_tensor=True)
        cosine_scores = util.cos_sim(k_embeddings, p_embeddings)

        for i, k_text in enumerate(k_titles):
            best_idx = cosine_scores[i].argmax().item()
            semantic_score = cosine_scores[i][best_idx].item()
            
            if semantic_score >= threshold:
                p_text = p_titles[best_idx]
                
                # 2. Key/Title Similarity (Fuzzy check on the specific text)
                key_sim = fuzz.token_sort_ratio(k_text, p_text) / 100.0
                
                # 3. Simple Average Score (The Balanced Model)
                total_score = round((semantic_score + key_sim) / 2, 4)

                all_results.append({
                    f'{type_label}_ids': {
                        'kalshi': k_subset.iloc[i][id_col],
                        'poly': p_subset.iloc[best_idx][id_col]
                    },
                    'kalshi_text': k_text,
                    'poly_text': p_text,
                    
                    # Score Breakdown for Testing
                    'semantic_score': round(semantic_score, 4),
                    'key_sim_score': round(key_sim, 4),
                    'total_score': total_score,
                    
                    # Keywords for Debugging
                    'key_word': {
                        'kalshi': str(k_subset.iloc[i]['match_key']),
                        'poly': str(p_subset.iloc[best_idx]['match_key'])
                    }
                })

    if not all_results:
        return pd.DataFrame()
        
    return pd.DataFrame(all_results).sort_values(by='total_score', ascending=False).reset_index(drop=True)

## --- 2. Data Hydration & Cleaning ---

def hydrate_and_clean_markets(match_df, event_id_col='event_ids'):
    """Fetches raw market data and applies the cleaning function immediately."""
    ids_list = match_df[event_id_col].tolist()
    poly_ids = [d.get('poly') for d in ids_list if d.get('poly')]
    kalshi_tickers = [d.get('kalshi') for d in ids_list if d.get('kalshi')]

    poly_rows, kalshi_rows = [], []

    # Polymarket Fetch
    if poly_ids:
        try:
            url = "https://gamma-api.polymarket.com/events"
            response = requests.get(url, params=[('id', eid) for eid in poly_ids], timeout=10)
            if response.status_code == 200:
                for event in response.json():
                    for m in event.get('markets', []):
                        # Construct descriptive text for matching
                        outcomes = str(m.get('outcomes', '')).replace("[", "").replace("]", "").replace("'", "")
                        poly_rows.append({
                            'event_id': str(event.get('id')),
                            'market_id': m.get('id'),
                            'descriptive_text': f"{m.get('question', '')} {outcomes}"
                        })
        except Exception as e: print(f"Poly Hydration Error: {e}")

    # Kalshi Fetch
    for ticker in kalshi_tickers:
        try:
            url = f"https://api.elections.kalshi.com/trade-api/v2/events/{ticker}?with_nested_markets=true"
            res = requests.get(url, timeout=5)
            if res.status_code == 200:
                event_data = res.json().get('event', {})
                for m in event_data.get('markets', []):
                    kalshi_rows.append({
                        'event_id': ticker,
                        'market_id': m.get('ticker'),
                        'descriptive_text': m.get('title', '')
                    })
        except Exception as e: print(f"Kalshi Hydration Error: {e}")

    # Convert to DataFrames and apply your custom cleaning
    df_p = pd.DataFrame(poly_rows)
    df_k = pd.DataFrame(kalshi_rows)

    if not df_p.empty:
        df_p = clean_text_cols(df_p, exclude_cols=['market_id', 'event_id'])
        df_p = create_match_keys(df_p, ['descriptive_text'])
    
    if not df_k.empty:
        df_k = clean_text_cols(df_k, exclude_cols=['market_id', 'event_id'])
        df_k = create_match_keys(df_k, ['descriptive_text'])

    return df_k, df_p



In [20]:
## --- 3. Execution Flow ---

# Fetch & Prep Events
kalshi.fetch_data()
poly.fetch_events()
_, k_events = kalshi.get_separated_dfs()
_, p_events = poly.get_separated_dfs()

Fetching Kalshi events...
Fetching Kalshi markets...
Fetching Polymarket events...


In [30]:


k_events = create_match_keys(k_events, ['event_title'])
p_events = create_match_keys(p_events, ['event_title'])

# Step 1: Match Events
matched_events = get_matches(
    k_events,
    p_events,
    'event_title',
    'event_id',
    type_label='event', 
    threshold=0.00
)
matched_events['key_word'].to_dict()

{0: {'kalshi': '2026_march_temperature_2026',
  'poly': '2026_march_temperature_2026'},
 1: {'kalshi': '2026_march_temperature_2026',
  'poly': '2026_march_temperature_2026'},
 2: {'kalshi': '2026_2026', 'poly': '2026_2026'},
 3: {'kalshi': '2026_2026', 'poly': '2026_2026'},
 4: {'kalshi': '2026_march_2026', 'poly': '2026_march_2026'},
 5: {'kalshi': '2026_2026', 'poly': '2026_2026'},
 6: {'kalshi': '2026_2026', 'poly': '2026_2026'},
 7: {'kalshi': '2026_2026', 'poly': '2026_2026'},
 8: {'kalshi': '2026_march_2026', 'poly': '2026_march_2026'},
 9: {'kalshi': '2026_2026', 'poly': '2026_2026'},
 10: {'kalshi': '2026_march_2026', 'poly': '2026_march_2026'},
 11: {'kalshi': '2026_march_2026', 'poly': '2026_march_2026'},
 12: {'kalshi': '2026_2026', 'poly': '2026_2026'},
 13: {'kalshi': '2026_2026', 'poly': '2026_2026'},
 14: {'kalshi': '2026_march_2026', 'poly': '2026_march_2026'},
 15: {'kalshi': '2026_2026', 'poly': '2026_2026'},
 16: {'kalshi': '2026_march_2026', 'poly': '2026_march_202

In [19]:
matched_events

""


In [18]:

# Step 2: Hydrate & Clean Markets for those matched events
k_markets_clean, p_markets_clean = hydrate_and_clean_markets(matched_events)

# Step 3: Match Markets
matched_markets = get_matches(
    k_markets_clean, 
    p_markets_clean, 
    'descriptive_text', 
    'market_id', 
    type_label='market', 
    threshold=0.70
)

display(matched_markets)

KeyError: 'event_ids'

In [ ]:
matched_markets[matched_markets['total_score'] > 0.75]
matched_events

,event_ids,kalshi_text,poly_text,total_score,match_keys
0,"{'kalshi': 'KXAHLGAME-26APR011900PROUTI', 'pol...",providence bruins vs utica comets,ahl providence bruins vs utica comets,0.9500,"{'kalshi': 'bruins_providence_2026', 'poly': '..."
1,"{'kalshi': 'KXAHLGAME-26MAR311900MANTOR', 'pol...",manitoba moose vs toronto marlies,ahl manitoba moose vs toronto marlies,0.9448,"{'kalshi': 'toronto_2026', 'poly': 'toronto_99..."
2,"{'kalshi': 'KXAHLGAME-26APR011030HERBRI', 'pol...",hershey bears vs bridgeport islanders,ahl hershey bears vs bridgeport islanders,0.9374,"{'kalshi': 'bears_bridgeport_2026', 'poly': 'b..."
3,"{'kalshi': 'KXHIGHNY-26MAR30', 'poly': '313358'}",highest temperature in nyc on march 30 2026,highest temperature in nyc on march 31,0.9235,"{'kalshi': '2026_march_temperature_2026', 'pol..."
4,"{'kalshi': 'KXAHLGAME-26APR011905CHAROC', 'pol...",charlotte checkers vs rochester americans,ahl charlotte checkers vs rochester americans,0.9224,"{'kalshi': 'charlotte_rochester_2026', 'poly':..."
5,"{'kalshi': 'KXAHLGAME-26MAR312200TEXSAN', 'pol...",texas stars vs san jose barracuda,ahl texas stars vs san jose barracuda,0.9127,"{'kalshi': 'jose_san_texas_2026', 'poly': 'jos..."
6,"{'kalshi': 'KXNEXTMANAGEREPL-26TOT', 'poly': '...",be the next manager of tottenham,next tottenham manager,0.8708,"{'kalshi': 'next_2026', 'poly': 'next_9999'}"
7,"{'kalshi': 'KXLOLMAP-26MAR31WEBLG-1', 'poly': ...",esports world cup china qualifier 2026 team we...,lol bilibili gaming vs team we bo3 esports wor...,0.8383,"{'kalshi': '2026_china_2026', 'poly': 'china_9..."
8,"{'kalshi': 'KXLOLMAP-26MAR31WEBLG-2', 'poly': ...",esports world cup china qualifier 2026 team we...,lol bilibili gaming vs team we bo3 esports wor...,0.8382,"{'kalshi': '2026_china_2026', 'poly': 'china_9..."
9,"{'kalshi': 'KXCS2MAP-26MAR30BLUNITY-1', 'poly'...",esl challenger league europe cup 3 2026 brazyl...,counter strike unity esports vs brazylijski lu...,0.8347,"{'kalshi': '2026_league_2026', 'poly': 'league..."
